In [1]:
import numpy as np
import torch
import os
import glob
import awkward as ak
import numpy as np
import pandas as pd

### Split MC events

In [ ]:
mc_path = "/home/aegis/Titan1/NRAD/data/Regions"
data_path = "/home/aegis/Titan1/NRAD/data/Regions_data"

In [3]:
# Load all mc events
print("Loading MC events...")

file_pattern_CR = os.path.join(mc_path, "CR*", "*.parquet")
file_pattern_SR = os.path.join(mc_path, "SR*", "*.parquet")
files_CR = glob.glob(file_pattern_CR)
files_SR = glob.glob(file_pattern_SR)

print(f"Found {len(files_CR)} files in Control Regions.")
print(f"Found {len(files_SR)} files in Signal Regions.")

SCALAR_VARS = ["met_recalc_pt", "met_recalc_phi", "weight_phys"]
VECTOR_VARS = ["AnalysisJetsAuxDyn_pt", "AnalysisJetsAuxDyn_eta", "AnalysisJetsAuxDyn_phi", 
               "AnalysisLargeRJetsAuxDyn_pt", "AnalysisLargeRJetsAuxDyn_eta", "AnalysisLargeRJetsAuxDyn_phi",
               "AnalysisLargeRJetsAuxDyn_Tau1_wta", "AnalysisLargeRJetsAuxDyn_Tau2_wta", "AnalysisLargeRJetsAuxDyn_Tau3_wta", 
               ]

columns_to_load = SCALAR_VARS + VECTOR_VARS
events_CR = ak.from_parquet(files_CR, columns=columns_to_load)
events_SR = ak.from_parquet(files_SR, columns=columns_to_load)
print(f"Loaded {len(events_CR)} events in Control Regions.")
print(f"Loaded {len(events_SR)} events in Signal Regions.")

Loading MC events...
Found 31752 files in Control Regions.
Found 5024 files in Signal Regions.
Loaded 5708123 events in Control Regions.
Loaded 154504 events in Signal Regions.


In [4]:
def delta_phi(phi1, phi2):
    # The standard "wrap-around" logic
    dphi = (phi1 - phi2 + np.pi) % (2 * np.pi) - np.pi
    return dphi
def prepare_events(events):

    jets = ak.zip({
        "pt": events["AnalysisJetsAuxDyn_pt"],
        "eta": events["AnalysisJetsAuxDyn_eta"],
        "phi": events["AnalysisJetsAuxDyn_phi"],
        "mass": ak.zeros_like(events["AnalysisJetsAuxDyn_pt"])
    }, with_name="Momentum4D")

    ljs = ak.zip({
        "pt": events["AnalysisLargeRJetsAuxDyn_pt"],
        "tau1": events["AnalysisLargeRJetsAuxDyn_Tau1_wta"],
        "tau2": events["AnalysisLargeRJetsAuxDyn_Tau2_wta"],
        "tau3": events["AnalysisLargeRJetsAuxDyn_Tau3_wta"]
    })

    met = ak.zip({
        "pt": events["met_recalc_pt"],
        "phi": events["met_recalc_phi"]
    }, with_name="Momentum2D")


    events["ht"] = ak.sum(events["AnalysisJetsAuxDyn_pt"], axis=1)/1000

    j_lead = jets[:, 0]
    j_sub  = jets[:, 1]

    events["mjj"] = np.sqrt(2 * j_lead.pt * j_sub.pt * (np.cosh(j_lead.eta - j_sub.eta) - np.cos(j_lead.phi - j_sub.phi)))/1000

    met_phi = events["met_recalc_phi"]
    # We need to broadcast MET phi to match the shape of the jet array for subtraction
    # e.g., if Event 1 has 3 jets, we need [met_phi, met_phi, met_phi]
    jet_phi = events["AnalysisJetsAuxDyn_phi"]
    met_phi_broadcasted = ak.broadcast_arrays(met_phi, jet_phi)[0]

    # B. Calculate dPhi between ALL jets and MET
    dphi_jet_met = np.abs(delta_phi(jet_phi, met_phi_broadcasted))

    # --- Save Variable: Min dPhi(Jet, MET) ---
    events["min_dphi_jet_met"] = ak.min(dphi_jet_met, axis=1)

    idx_closest = ak.argmin(dphi_jet_met, axis=1, keepdims=True)
    idx_farthest = ak.argmax(dphi_jet_met, axis=1, keepdims=True)

    # D. Extract pt and phi for these specific jets
    # syntax: array[indices] gives us the specific values
    pt_closest = events["AnalysisJetsAuxDyn_pt"][idx_closest][:, 0]
    phi_closest = events["AnalysisJetsAuxDyn_phi"][idx_closest][:, 0]

    pt_farthest = events["AnalysisJetsAuxDyn_pt"][idx_farthest][:, 0]
    phi_farthest = events["AnalysisJetsAuxDyn_phi"][idx_farthest][:, 0]

    # E. Calculate Vector Sum of the two jets (Numerator)
    # We add their x and y components
    vec_sum_px = (pt_closest * np.cos(phi_closest)) + (pt_farthest * np.cos(phi_farthest))
    vec_sum_py = (pt_closest * np.sin(phi_closest)) + (pt_farthest * np.sin(phi_farthest))
    vec_sum_pt = np.sqrt(vec_sum_px**2 + vec_sum_py**2)

    # F. Calculate Scalar Sum (Denominator)
    scalar_sum_pt = pt_closest + pt_farthest

    # G. Final Calculation
    # events["pt_balance"] = vec_sum_pt / scalar_sum_pt
    denom_safe = scalar_sum_pt > 0
    raw_balance = vec_sum_pt / scalar_sum_pt

    # Use ak.where to replace the bad values with 0
    # Syntax: ak.where(condition, value_if_true, value_if_false)
    events["pt_balance"] = ak.where(denom_safe, raw_balance, 0)

    events["dphi_j1_j2"] = np.abs(delta_phi(phi_closest, phi_farthest))

    # Leading Large-R Jet (Index 0)
    tau1_lead = events["AnalysisLargeRJetsAuxDyn_Tau1_wta"][:, 0]
    tau2_lead = events["AnalysisLargeRJetsAuxDyn_Tau2_wta"][:, 0]
    tau3_lead = events["AnalysisLargeRJetsAuxDyn_Tau3_wta"][:, 0]

    events["ljet1_tau21"] = ak.where(tau1_lead > 0, tau2_lead / tau1_lead, 0)
    events["ljet1_tau32"] = ak.where(tau2_lead > 0, tau3_lead / tau2_lead, 0)

    # Subleading Large-R Jet (Index 1)
    tau1_sub = events["AnalysisLargeRJetsAuxDyn_Tau1_wta"][:, 1]
    tau2_sub = events["AnalysisLargeRJetsAuxDyn_Tau2_wta"][:, 1]
    tau3_sub = events["AnalysisLargeRJetsAuxDyn_Tau3_wta"][:, 1]

    events["ljet2_tau21"] = ak.where(tau1_sub > 0, tau2_sub / tau1_sub, 0)
    events["ljet2_tau32"] = ak.where(tau2_sub > 0, tau3_sub / tau2_sub, 0)

    print("Calculation complete. New variables added to 'events'.")

    return events

In [5]:
events_CR = prepare_events(events_CR)
events_SR = prepare_events(events_SR)

/home/aegis/.local/lib/python3.10/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


Calculation complete. New variables added to 'events'.
Calculation complete. New variables added to 'events'.


In [6]:
# List of new High-Level Variables (HLVs) you just created
NEW_HLVS = [
    "ht",
    "met_recalc_pt", 
    "mjj", 
    "pt_balance", 
    "dphi_j1_j2", 
    "ljet1_tau21", "ljet1_tau32", 
    "ljet2_tau21", "ljet2_tau32", 
    "min_dphi_jet_met", 
    "weight_phys"
]
df_hlvs_CR = ak.to_dataframe(events_CR[NEW_HLVS])
df_hlvs_SR = ak.to_dataframe(events_SR[NEW_HLVS])

df_hlvs_CR['Region'] = 'CR'
df_hlvs_SR['Region'] = 'SR'

df_combined = pd.concat([df_hlvs_CR, df_hlvs_SR], ignore_index=True)

In [ ]:
print(f"NaNs before cleaning:\n{df_combined.isna().sum()}")
print(f"Infinities before cleaning:\n{np.isinf(df_combined[NEW_HLVS]).sum()}")
print("Cleaning data...")

df_clean = df_combined.fillna(0.0)
# Handle Infs (just in case division by zero slipped through)
df_clean = df_clean.replace([np.inf, -np.inf], 0.0)

NaNs before cleaning:
ht                  0
met_recalc_pt       0
mjj                 0
pt_balance          0
dphi_j1_j2          0
ljet1_tau21         0
ljet1_tau32         0
ljet2_tau21         0
ljet2_tau32         0
min_dphi_jet_met    0
weight_phys         0
Region              0
dtype: int64
Infinities before cleaning:
ht                  0
met_recalc_pt       0
mjj                 0
pt_balance          0
dphi_j1_j2          0
ljet1_tau21         0
ljet1_tau32         0
ljet2_tau21         0
ljet2_tau32         0
min_dphi_jet_met    0
weight_phys         0
dtype: int64
Cleaning data...


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib  # For saving the scaler

X = df_clean

print("Splitting data (75% Train, 25% Test)...")
X_train, X_test = train_test_split(X, test_size=0.25, random_state=42, shuffle=True)

print(f"Train shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")

# === 3. SCALE (Fit on Train ONLY) ===
print("Scaling features...")
scaler = MinMaxScaler(feature_range=(-2.5, 2.5))

# A. FIT only on Training data
NEW_HLVS_SUB = [v for v in NEW_HLVS if v != 'weight_phys']

scaler.fit(X_train[NEW_HLVS_SUB])

# # B. TRANSFORM both Train and Test
# X_train_scaled_CR = scaler.transform(X_train[X_train["Region"] == "CR"][NEW_HLVS_SUB])
# X_train_scaled_SR = scaler.transform(X_train[X_train["Region"] == "SR"][NEW_HLVS_SUB])

# X_test_scaled_CR = scaler.transform(X_test[X_test["Region"] == "CR"][NEW_HLVS_SUB])
# X_test_scaled_SR = scaler.transform(X_test[X_test["Region"] == "SR"][NEW_HLVS])

X_train_df_CR = X_train[X_train["Region"] == "CR"].copy()
X_train_df_SR = X_train[X_train["Region"] == "SR"].copy()

X_test_df_CR = X_test[X_test["Region"] == "CR"].copy()
X_test_df_SR = X_test[X_test["Region"] == "SR"].copy()

# X_train_scaled = scaler.transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# C. Convert back to DataFrame (Scaler returns numpy array)
# We keep the column names for clarity
# X_train_df_CR = pd.DataFrame(X_train_scaled_CR, columns=NEW_HLVS)
# X_train_df_SR = pd.DataFrame(X_train_scaled_SR, columns=NEW_HLVS)

# X_test_df_CR = pd.DataFrame(X_test_scaled_CR, columns=NEW_HLVS)
# X_test_df_SR = pd.DataFrame(X_test_scaled_SR, columns=NEW_HLVS)

X_train_df_CR[NEW_HLVS_SUB] = scaler.transform(X_train_df_CR[NEW_HLVS_SUB])
X_train_df_SR[NEW_HLVS_SUB] = scaler.transform(X_train_df_SR[NEW_HLVS_SUB])

X_test_df_CR[NEW_HLVS_SUB] = scaler.transform(X_test_df_CR[NEW_HLVS_SUB])
X_test_df_SR[NEW_HLVS_SUB] = scaler.transform(X_test_df_SR[NEW_HLVS_SUB])

Splitting data (75% Train, 25% Test)...
Train shape: (4396970, 12)
Test shape:  (1465657, 12)
Scaling features...


In [16]:
print("Converting to float32 for efficiency...")
X_train_df_CR = X_train_df_CR.drop(columns=["Region"]).astype(np.float32)
X_train_df_SR = X_train_df_SR.drop(columns=["Region"]).astype(np.float32)
X_test_df_CR = X_test_df_CR.drop(columns=["Region"]).astype(np.float32)
X_test_df_SR = X_test_df_SR.drop(columns=["Region"]).astype(np.float32)

# === 5. SAVE EVERYTHING ===
output_dir = "Scaled_Regions_with_weights"
os.makedirs(output_dir, exist_ok=True)

print(f"Saving to {output_dir}...")

X_train_df_CR.to_parquet(os.path.join(output_dir, "MC_CR_train.parquet"), compression="zstd")
X_test_df_CR.to_parquet(os.path.join(output_dir, "MC_CR_test.parquet"), compression="zstd")

X_train_df_SR.to_parquet(os.path.join(output_dir, "MC_SR_train.parquet"), compression="zstd")
X_test_df_SR.to_parquet(os.path.join(output_dir, "MC_SR_test.parquet"), compression="zstd")

scaler_filename = os.path.join(output_dir, "scaler_cr.save")
joblib.dump(scaler, scaler_filename)

print("="*40)
print("Pipeline Complete.")
print(f"3. Scaler Object:   {scaler_filename}")
print("="*40)

Converting to float32 for efficiency...


Saving to Scaled_Regions_with_weights...
Pipeline Complete.
3. Scaler Object:   Scaled_Regions_with_weights/scaler_cr.save


### Split Data events

In [ ]:
# Load all mc events
print("Loading Data events...")

file_pattern_CR = os.path.join(data_path, "CR*", "*.parquet")
file_pattern_SR = os.path.join(data_path, "SR*", "*.parquet")
files_CR = glob.glob(file_pattern_CR)
files_SR = glob.glob(file_pattern_SR)

print(f"Found {len(files_CR)} files in Control Regions.")
print(f"Found {len(files_SR)} files in Signal Regions.")

SCALAR_VARS = ["met_recalc_pt", "met_recalc_phi", "weight_phys"]
VECTOR_VARS = ["AnalysisJetsAuxDyn_pt", "AnalysisJetsAuxDyn_eta", "AnalysisJetsAuxDyn_phi", 
               "AnalysisLargeRJetsAuxDyn_pt", "AnalysisLargeRJetsAuxDyn_eta", "AnalysisLargeRJetsAuxDyn_phi",
               "AnalysisLargeRJetsAuxDyn_Tau1_wta", "AnalysisLargeRJetsAuxDyn_Tau2_wta", "AnalysisLargeRJetsAuxDyn_Tau3_wta", 
               ]
# print(files)

columns_to_load = SCALAR_VARS + VECTOR_VARS
events_data_CR = ak.from_parquet(files_CR, columns=columns_to_load)
events_data_SR = ak.from_parquet(files_SR, columns=columns_to_load)
print(f"Loaded {len(events_data_CR)} events in Control Regions.")
print(f"Loaded {len(events_data_SR)} events in Signal Regions.")

Loading Data events...
Found 225031 files in Control Regions.
Found 27111 files in Signal Regions.


In [42]:
events_data_CR = prepare_events(events_data_CR)
events_data_SR = prepare_events(events_data_SR)

/home/aegis/.local/lib/python3.10/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


Calculation complete. New variables added to 'events'.
Calculation complete. New variables added to 'events'.


In [43]:
# List of new High-Level Variables (HLVs) you just created
NEW_HLVS = [
    "ht",
    "met_recalc_pt", 
    "mjj", 
    "pt_balance", 
    "dphi_j1_j2", 
    "ljet1_tau21", "ljet1_tau32", 
    "ljet2_tau21", "ljet2_tau32", 
    "min_dphi_jet_met"
]

df_hlvs_data_CR = ak.to_dataframe(events_data_CR[NEW_HLVS])
df_hlvs_data_SR = ak.to_dataframe(events_data_SR[NEW_HLVS])
# df_hlvs_data_CR.head()

df_hlvs_data_CR['Region'] = 'CR'
df_hlvs_data_SR['Region'] = 'SR'

df_data_combined = pd.concat([df_hlvs_data_CR, df_hlvs_data_SR], ignore_index=True)
df_data_combined

,ht,met_recalc_pt,mjj,pt_balance,dphi_j1_j2,ljet1_tau21,ljet1_tau32,ljet2_tau21,ljet2_tau32,min_dphi_jet_met,Region
0,564.524170,251.540237,536.193420,0.819899,1.783257,0.628249,0.689035,0.293643,0.668827,0.249897,CR
1,463.338196,290.407257,404.102142,0.272282,3.069904,0.696639,0.787107,0.620565,0.674627,0.124865,CR
2,553.819458,295.911072,471.982727,0.635773,2.889998,0.643989,0.479774,0.726980,0.864179,0.024814,CR
3,682.811890,294.443420,503.701019,0.155354,2.880652,0.617668,0.816805,0.437166,0.356303,0.035141,CR
4,662.238892,284.867493,720.650940,0.552377,2.245857,0.451966,0.749601,0.544761,0.747789,0.721098,CR
...,...,...,...,...,...,...,...,...,...,...,...
53099325,979.845093,600.928772,624.988708,0.410500,2.714305,0.220706,0.892273,0.385978,0.750554,0.354400,SR
53099326,1364.303711,618.584412,1520.226929,0.192283,3.121555,0.250428,0.867493,0.352826,0.859473,0.023988,SR
53099327,835.690491,723.249634,433.194855,0.813578,2.962935,0.391412,0.264582,0.413799,0.658943,0.209204,SR
53099328,837.260193,602.526917,938.065918,0.203152,2.732443,0.595389,0.725397,0.459708,0.677437,0.325223,SR


In [44]:
print(f"NaNs before cleaning:\n{df_data_combined.isna().sum()}")
print(f"Infinities before cleaning:\n{np.isinf(df_data_combined[NEW_HLVS]).sum()}")
print("Cleaning data...")

df_clean_data = df_data_combined.fillna(0.0)
# Handle Infs (just in case division by zero slipped through)
df_clean_data = df_clean_data.replace([np.inf, -np.inf], 0.0)

NaNs before cleaning:
ht                  0
met_recalc_pt       0
mjj                 0
pt_balance          0
dphi_j1_j2          0
ljet1_tau21         0
ljet1_tau32         0
ljet2_tau21         0
ljet2_tau32         0
min_dphi_jet_met    0
Region              0
dtype: int64
Infinities before cleaning:
ht                  0
met_recalc_pt       0
mjj                 0
pt_balance          0
dphi_j1_j2          0
ljet1_tau21         0
ljet1_tau32         0
ljet2_tau21         0
ljet2_tau32         0
min_dphi_jet_met    0
dtype: int64
Cleaning data...


In [45]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib  # For saving the scaler

X_data = df_clean_data

print("Splitting data (75% Train, 25% Test)...")
X_train_data, X_test_data = train_test_split(X_data, test_size=0.25, random_state=42, shuffle=True)


loaded_scaler = joblib.load("Scaled_Regions/scaler_cr.save")

print(f"Train shape: {X_train_data.shape}")
print(f"Test shape:  {X_test_data.shape}")

# === 3. SCALE (Fit on Train ONLY) ===
print("Scaling features...")

X_train_scaled_CR_data = loaded_scaler.transform(X_train_data[X_train_data["Region"] == "CR"][NEW_HLVS])
X_train_scaled_SR_data = loaded_scaler.transform(X_train_data[X_train_data["Region"] == "SR"][NEW_HLVS])

X_test_scaled_CR_data = loaded_scaler.transform(X_test_data[X_test_data["Region"] == "CR"][NEW_HLVS])
X_test_scaled_SR_data = loaded_scaler.transform(X_test_data[X_test_data["Region"] == "SR"][NEW_HLVS])
# X_train_scaled_data = loaded_scaler.transform(X_train_data)
# X_test_scaled_data = loaded_scaler.transform(X_test_data)

# X_train_df_data = pd.DataFrame(X_train_scaled_data, columns=X_train_data.columns)
# X_test_df_data = pd.DataFrame(X_test_scaled_data, columns=X_test_data.columns)

X_train_df_CR_data = pd.DataFrame(X_train_scaled_CR_data, columns=NEW_HLVS)
X_train_df_SR_data = pd.DataFrame(X_train_scaled_SR_data, columns=NEW_HLVS)

X_test_df_CR_data = pd.DataFrame(X_test_scaled_CR_data, columns=NEW_HLVS)
X_test_df_SR_data = pd.DataFrame(X_test_scaled_SR_data, columns=NEW_HLVS)

print("Converting to float32 for efficiency...")

X_train_df_CR_data = X_train_df_CR_data.astype(np.float32)
X_train_df_SR_data = X_train_df_SR_data.astype(np.float32)
X_test_df_CR_data = X_test_df_CR_data.astype(np.float32)
X_test_df_SR_data = X_test_df_SR_data.astype(np.float32)

output_dir_data = "Scaled_Regions"
os.makedirs(output_dir_data, exist_ok=True)
print(f"Saving to {output_dir_data}...")

X_train_df_CR_data.to_parquet(os.path.join(output_dir_data, "Data_CR_train.parquet"), compression="zstd")
X_test_df_CR_data.to_parquet(os.path.join(output_dir_data, "Data_CR_test.parquet"), compression="zstd")

X_train_df_SR_data.to_parquet(os.path.join(output_dir_data, "Data_SR_train.parquet"), compression="zstd")
X_test_df_SR_data.to_parquet(os.path.join(output_dir_data, "Data_SR_test.parquet"), compression="zstd")

print("="*40)
print("Pipeline Complete.")
print("="*40)

Splitting data (75% Train, 25% Test)...
Train shape: (39824497, 11)
Test shape:  (13274833, 11)
Scaling features...
Converting to float32 for efficiency...
Saving to Scaled_Regions...
Pipeline Complete.
